# TFM7 · Asistente experto en informática y ciberseguridad

El sistema fundamenta sus respuestas en publicaciones NIST. Instala `requirements.txt` en el entorno del kernel. Copia `.env.example` a `.env` y configura `GEMINI_API_KEY` antes de ejecutar la indexación.

In [ ]:
from uuid import uuid4
from rag import chunks, open_index, retrieve, build_agent, converse, UNKNOWN
from IPython.display import display, Markdown

passages = list(chunks())
assert len({metadata['file'] for _, metadata in passages}) >= 3
assert len({(m['file'], m['page']) for _, m in passages}) >= 20
print(f'{len(passages)} fragmentos extraídos')

## Indexación y recuperación

La primera ejecución genera embeddings de los fragmentos. Las siguientes reutilizan la colección persistida.

In [ ]:
index = open_index()
assert index[0].count() == len(passages)
for question, expected in [('What are the six CSF 2.0 Functions?', 'nist-csf.pdf'), ('What are the basic tenets of zero trust?', 'nist-207.pdf'), ('What is the incident response life cycle?', 'nist-61r3.pdf')]:
    hits = retrieve(index, question)
    print(question, [(h['file'], h['page'], round(h['distance'], 3)) for h in hits])
    assert expected in {h['file'] for h in hits}

## Cinco preguntas y respuestas de referencia

Estas respuestas son síntesis de las publicaciones. La siguiente celda genera y muestra las respuestas reales con sus citas al configurar Gemini.

| Pregunta | Respuesta de referencia | Fuente |
|---|---|---|
| ¿Cuáles son las seis funciones de CSF 2.0? | Govern, Identify, Protect, Detect, Respond y Recover. | NIST CSF 2.0, §2.1 |
| ¿Qué representa un perfil objetivo del CSF? | Los resultados de ciberseguridad que la organización desea alcanzar. | NIST CSF 2.0, §3.1 |
| ¿Qué implica la ubicación de red en Zero Trust? | La ubicación por sí sola no concede confianza implícita a los usuarios o activos. | SP 800-207, §2 |
| ¿Qué hace el punto de aplicación de políticas en Zero Trust? | Habilita, supervisa y termina conexiones entre el sujeto y el recurso. | SP 800-207, §3 |
| ¿Qué funciones abarcan el ciclo operativo de respuesta a incidentes? | Detect, Respond y Recover. | SP 800-61 Rev. 3, §2 |


In [ ]:
agent = build_agent(index)
questions = [
    "¿Cuáles son las seis funciones de CSF 2.0?",
    "¿Qué representa un perfil objetivo del CSF?",
    "¿Qué implica la ubicación de red en Zero Trust?",
    "¿Qué hace el punto de aplicación de políticas en Zero Trust?",
    "¿Qué funciones abarcan el ciclo operativo de respuesta a incidentes?",
]

def show(result):
    display(Markdown(result["answer"]))
    for source in result["sources"]:
        print(f"[{source['number']}] {source['title']}, PDF p. {source['page']} · {source['url']}")

for question in questions:
    print(question)
    show(converse(agent, question, str(uuid4())))


## Memoria y ausencia de evidencia

La segunda pregunta omite el tema. Comprueba la consulta reformulada y las citas de la respuesta. Un identificador nuevo aísla otra conversación.

In [ ]:
thread = str(uuid4())
show(converse(agent, "¿Qué hace el punto de aplicación de políticas en Zero Trust?", thread))
followup = converse(agent, "¿En qué componentes se puede dividir?", thread)
print(followup["query"])
show(followup)
assert "zero trust" in followup["query"].lower() or "política" in followup["query"].lower()
assert any(s["file"] == "nist-207.pdf" for s in followup["sources"])
assert len(followup["history"]) == 2
fresh = converse(agent, "¿Cuál es la receta de la paella valenciana?", str(uuid4()))
assert len(fresh["history"]) == 1
assert fresh["answer"] == UNKNOWN
show(fresh)

## Chat interactivo

Escribe `salir` para terminar. Cada ejecución inicia una conversación.

In [ ]:
chat_id = str(uuid4())
while True:
    question = input("Tú: ").strip()
    if question.lower() == "salir":
        break
    if question:
        show(converse(agent, question, chat_id))